In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import timm
from torch import nn
from sklearn.metrics import ( accuracy_score, confusion_matrix, classification_report, precision_score, recall_score, f1_score )
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib as mpl
import sys
from galaxy_datasets.pytorch.galaxy_datamodule import CatalogDataModule
from galaxy_datasets.transforms import ( minimal_view_config, get_galaxy_transform )


In [ ]:
seed_value = 1000
np.random.seed(seed_value)
torch.manual_seed(seed_value)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
image_size = 224
batch_size = 1

BASE_OUTPUT_DIR = "outputs"
RUN_NAME = "train_10"   # <<< CHANGE ONLY THIS

RUN_DIR = os.path.join(BASE_OUTPUT_DIR, RUN_NAME)
os.makedirs(RUN_DIR, exist_ok=True)

# Update paths automatically
test_catalog_path = os.path.join(RUN_DIR, "test_catalog.csv")
model_path = os.path.join(RUN_DIR, "BEST_model.pt")

In [ ]:
class CustomHead(nn.Module):
    def __init__(self, in_channels, num_classes):
        super().__init__()
        self.global_avg_pool = nn.AdaptiveAvgPool2d(1)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(in_channels, 128)
        self.drop1 = nn.Dropout(0.2)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.global_avg_pool(x)
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.drop1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

class EfficientNetClassifier(nn.Module):
    def __init__(self, encoder, num_classes):
        super().__init__()
        self.encoder = encoder
        self.head = CustomHead(in_channels, num_classes)

    def forward(self, x):
        x = self.encoder(x)[-1]
        x = self.head(x)
        return x

In [ ]:
test_catalog = pd.read_csv(test_catalog_path)
print(f"Loaded test catalog: {len(test_catalog)} samples")

warp_map = {"WP": 0, "ED": 1}
inv_warp = {v: k for k, v in warp_map.items()}

In [ ]:
cfg = minimal_view_config()
cfg.output_size = image_size
cfg.pil_to_tensor = False
cfg.flux_to_jpg_like_dynamic_range = {
    "arcsinh_q": 1.0,
    "percentile_min": 0,
    "percentile_max": 99.7
}
test_transform = get_galaxy_transform(cfg)

In [ ]:
datamodule = CatalogDataModule(
    test_catalog=test_catalog,
    batch_size=batch_size,
    label_cols=["label"],
    test_transform=test_transform
)
datamodule.setup(stage="test")
test_loader = datamodule.test_dataloader()

In [ ]:
encoder = timm.create_model(
    "hf_hub:mwalmsley/zoobot-encoder-convnext_nano",
    pretrained=True,
    features_only=True,
    in_chans=1
)

in_channels = encoder.feature_info[-1]["num_chs"]
model = EfficientNetClassifier(encoder, 2).to(device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()


In [ ]:
y_true, y_pred = [], []
rows = []

with torch.no_grad():
    for batch in test_loader:
        x = batch["image"].to(device)
        y = batch["label"].to(device)
        ids = batch["id_str"]

        logits = model(x)
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        for i in range(len(preds)):
            y_true.append(y[i].item())
            y_pred.append(preds[i].item())

            rows.append({
                "Obj_name": ids[i],
                "Warp_true": inv_warp[y[i].item()],
                "Warp_pred": inv_warp[preds[i].item()],
                "WP_prob": probs[i, 0].item(),
                "ED_prob": probs[i, 1].item()
            })
